## 一、可变形 DETR 概述

* 目标：解决原始 DETR 中 **收敛慢、对小目标不敏感** 的问题。
* 关键创新：

  * **多尺度特征金字塔**：无需FPN（Feature Pyramid Network），通过可变形注意力自然地聚合多尺度特征，增强了小物体检测的能力。
  * **可变形注意力（Deformable Attention）**：与传统Transformer的全局注意力不同，可变形DETR仅关注参考点周围的少数关键采样点，显著降低了计算成本

## 二、推理流程及代码运行逻辑

### 推理代码主流程（伪代码）：

```python
features = backbone(images)                        # 提取多尺度特征
multi_scale_features = input_proj(features)        # 统一通道数
pos_embeds = get_position_embedding(features)      # 每个尺度位置编码

memory = transformer.encoder(multi_scale_features, pos_embeds)
hs = transformer.decoder(query_embed, memory)      # 输出预测 embedding

outputs = FFN(hs)                                   # 分类 + 回归
```
### 2.1 可变形注意力机制公式
$$E_q= \sum_{k=1}^K A_{mqk} \cdot V_k$$
其中，$E_q$ 表示查询位置 q 的输出特征；$A_{mqk}$ 是第 m 个头对于第 k 个采样点的注意力权重；$V_k$ 是第 k 个采样点的特征值。通过这种方式，模型可以选择性地关注重要区域，而非遍历整个特征图。
### 2.2 多尺度特征聚合公式
$$F_l= \sum_{i=1}^L W_i \cdot f_i$$
这里 $F_l$ 是第 l 层的综合特征，$f_i$ 表示第 i 层的特征图，$W_i$ 是对应的权重系数。通过对不同层次的特征加权求和，实现了多尺度特征的有效融合。

## 三、Tensor 尺寸变化解析（关键维度跟踪）

假设图像输入大小为 (B, 3, H, W)，num_queries=300，num_feature_levels=4

| 模块                            | 尺寸                                                                       |
| ----------------------------- | ------------------------------------------------------------------------ |
| 输入图像                          | (B, 3, H, W)                                                          |
| CNN Backbone 输出 V经过线性变换              | [(B, C, H/8, W/8), ..., (B, C, H/32, W/32)] 共 4 个尺度                    |
| input\_proj                   | 每层统一成 (B, d_model, $H_i, W_i$)                                          |
| Flatten + Positional Encoding | 每层展平为 (B, $H_i*W_i$, d_model)                                          |
| encoder 输出                    | 多层合并后尺寸为 (B,$∑_i H_i*W_i$, d_model)                                 |
| Decoder 输入 query              | `(num_queries, d_model)`                                                 |
| Decoder 输出                    | `(B, num_decoder_layers, num_queries, d_model)`                          |
| FFN 预测                        | 分类 logits: `(B, num_queries, num_classes)`； boxes: `(B, num_queries, 4)` |

## 四、可变形 DETR 模型结构模块说明

### 1. **Backbone**

* 通常使用 ResNet-50/101
* 输出多尺度特征图（FPN）

### 2. **可变形注意力模块（Deformable Attention）**

**区别于普通注意力：**

* 普通 attention 在全部 token 上求和；
* Deformable Attention 仅在 **K 个偏移位置采样点** 上聚合（如每个 key 只看 4 个点）。

**图片通过BackBone，经过线性变换得到V**

**Q随机初始化可学习，经过线性层处理后，得到参考点sampling_offsets的偏移以及经softmax后采样点points的权重**

**权重**

结构流程：

```
query → sampling_offsets → (d x K) points → pos_abs
       ↓
    extract features from memory
       ↓
     aggregate with attention_weights
```

### 3. **Transformer（多尺度）**

#### 3.1 编码器（Encoder）
- 输入：图像经过CNN提取特征后得到的特征图。
- 处理：在编码器中，使用了多个可变形注意力模块代替标准的Transformer注意力模块。每个模块会根据参考点选择若干个采样点进行信息交互。
- 输出：生成的特征表示用于后续解码阶段。
#### 3.2 解码器（Decoder）
- 输入：来自编码器的特征表示以及一组预定义的查询位置（queries）。
- 处理：解码器同样采用了可变形注意力机制，通过多次迭代逐步精炼查询位置的内容，直到生成最终的预测结果。
- 输出：预测的对象类别和边界框坐标。
#### 3.3 查询位置（Queries）
- 初始化：查询位置通常被初始化为一组可学习的嵌入向量。
- 内容：这些嵌入包含了位置编码、类别偏好及多尺度信息等，有助于模型学习到不同物体的特征。

## 五、损失函数说明

同 DETR，使用 Hungarian Matching 匹配 ground truth 和预测。

损失由以下几部分组成：

| 类型      | 计算方式                                |
| ------- | ----------------------------------- |
| 分类损失    | CrossEntropy                        |
| L1 边框损失 | $\text{L1}(b_{pred}, b_{gt})$       |
| GIoU 损失 | $1 - \text{GIoU}(b_{pred}, b_{gt})$ |
| 匹配      | 匈牙利算法（匹配预测与 GT）                     |

损失函数公式：

```math
\mathcal{L} = \lambda_{\text{cls}} \cdot \mathcal{L}_{\text{cls}} + \lambda_{\text{L1}} \cdot \mathcal{L}_{\text{L1}} + \lambda_{\text{GIoU}} \cdot \mathcal{L}_{\text{GIoU}}
```

## 六、ROI 与可线性插值（特征采样）

### 1. **ROI（Region of Interest）理解**

* 可变形注意力中，query 生成一组采样偏移点 → 从特定 feature map 上采样；
* 本质上形成了动态的 ROI，每个 query 自适应地选择注意力区域。

### 2. **双线性插值（Bilinear Interpolation）**

* 用于从 feature map 上 **非整数位置提取特征**
* 给定浮点坐标位置 (x, y)，使用周围四点插值计算特征值



## 七、结论（总结）

| 项目       | 内容                                          |
| -------- | ------------------------------------------- |
| 改进点      | 多尺度特征 + 可变形注意力                              |
| 推理过程     | CNN → 特征金字塔 → Attention 聚合 → Transformer 解码 |
| Tensor变化 | 明确尺寸变换过程，注意 flatten 和 concat 位置             |
| 损失       | Hungarian Matching + L1 + GIoU              |
| 插值采样     | 可变形注意力采用 ROI + 双线性插值采样                      |